In [1]:
import sys
!{sys.executable} -m pip install hdfs

In [22]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable

# 1. Configuração da Sessão Spark
builder = SparkSession.builder \
    .appName("Gold_Dim_Paises_Unicos") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .enableHiveSupport()

spark = configure_spark_with_delta_pip(builder).getOrCreate()

# Caminhos
silver_path = "hdfs://hdfs-nn:9000/demo/silver"
gold_paises_path = "hdfs://hdfs-nn:9000/demo/gold/dim_paises"

# 2. Mapeamento Interno Consolidado (Evita duplicados manuais)
paises_data = [
    ("US", "UNITED STATES OF AMERICA"), ("GB", "UNITED KINGDOM"), ("IN", "INDIA"),
    ("FR", "FRANCE"), ("CA", "CANADA"), ("ES", "SPAIN"), ("DE", "GERMANY"),
    ("JP", "JAPAN"), ("BR", "BRAZIL"), ("PT", "PORTUGAL"), ("AU", "AUSTRALIA"),
    ("IT", "ITALY"), ("CN", "CHINA"), ("KR", "SOUTH KOREA"), ("MX", "MEXICO"),
    ("AE", "UNITED ARAB EMIRATES"), ("AF", "AFGHANISTAN"), ("AL", "ALBANIA"),
    ("AR", "ARGENTINA"), ("AT", "AUSTRIA"), ("BE", "BELGIUM"), ("CH", "SWITZERLAND"),
    ("CL", "CHILE"), ("CO", "COLOMBIA"), ("DK", "DENMARK"), ("EG", "EGYPT"),
    ("FI", "FINLAND"), ("GR", "GREECE"), ("HK", "HONG KONG"), ("HU", "HUNGARY"),
    ("ID", "INDONESIA"), ("IE", "IRELAND"), ("IL", "ISRAEL"), ("MY", "MALAYSIA"),
    ("NL", "NETHERLANDS"), ("NO", "NORWAY"), ("NZ", "NEW ZEALAND"), ("PH", "PHILIPPINES"),
    ("PL", "POLAND"), ("RO", "ROMANIA"), ("RU", "RUSSIAN FEDERATION"), ("SA", "SAUDI ARABIA"),
    ("SE", "SWEDEN"), ("SG", "SINGAPORE"), ("TH", "THAILAND"), ("TR", "TURKEY"),
    ("UA", "UKRAINE"), ("VN", "VIET NAM"), ("ZA", "SOUTH AFRICA"), ("NG", "NIGERIA")
]
df_map = spark.createDataFrame(paises_data, ["m_sigla", "m_nome"])

# 3. Função de Extração Silver (Delta)
def extrair_paises_silver(folder_name, coluna):
    path = f"{silver_path}/{folder_name}"
    try:
        df = spark.read.format("delta").load(path)
        # Limpa caracteres
        df = df.withColumn("p_clean", F.regexp_replace(F.col(coluna), r"[\[\]']", ""))
        # Separa países
        df = df.select(F.explode(F.split(F.col("p_clean"), r"[,|]")).alias("pais_raw"))
        return df.select(F.upper(F.trim(F.col("pais_raw"))).alias("pais_raw")).distinct()
    except Exception: return None

# Extrair das 3 fontes Silver
p_net = extrair_paises_silver("titles_delta", "production_countries")
p_ama = extrair_paises_silver("silver_titles", "production_countries")
p_mov = extrair_paises_silver("movies_silver", "production_countries")

dfs_validos = [df for df in [p_net, p_ama, p_mov] if df is not None]

if dfs_validos:
    df_all_raw = dfs_validos[0]
    for next_df in dfs_validos[1:]:
        df_all_raw = df_all_raw.union(next_df)
    
# 4. Lógica Anti-Duplicados e Apenas Nomes
dim_paises_pre = df_all_raw.join(df_map, (df_all_raw.pais_raw == df_map.m_sigla) | (df_all_raw.pais_raw == df_map.m_nome), "left") \
        .select(
            F.coalesce(F.col("m_sigla"), F.when(F.length(F.col("pais_raw")) <= 3, F.col("pais_raw")).otherwise(F.lit(None))).alias("sigla_temp"),
            F.coalesce(F.col("m_nome"), F.when(F.length(F.col("pais_raw")) > 3, F.col("pais_raw")).otherwise(F.lit(None))).alias("nome_temp")
        ).filter(F.col("nome_temp").isNotNull())

# Agrupamos por NOME para garantir que cada país aparece apenas uma vez
dim_paises_unique = dim_paises_pre.groupBy("nome_temp").agg(F.first("sigla_temp", ignorenulls=True).alias("sigla")).withColumnRenamed("nome_temp", "nome_completo")

# 5. Gerar ID Sequencial com prefixo "P"
window_spec = Window.orderBy("nome_completo")
dim_paises_gold = dim_paises_unique.withColumn("num", F.row_number().over(window_spec)).withColumn("id_pais", F.concat(F.lit("P"), F.col("num").cast("string"))).select("id_pais", "sigla", "nome_completo")

# 6. Gravação Gold e Limpeza
print(f"-> A Gravar Dimensão Países Únicos em {gold_paises_path}")
dim_paises_gold.repartition(1).write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .save(gold_paises_path)

print(f"\nDimensão Países Gold finalizada!")
dim_paises_gold.withColumn("n", F.regexp_replace("id_pais", "P", "").cast("int")).orderBy("n").drop("n").show(100,truncate=False)

spark.stop()

-> A Gravar Dimensão Países Únicos em hdfs://hdfs-nn:9000/demo/gold/dim_paises

Dimensão Países Gold finalizada!
+-------+-----+-------------------------------+
|id_pais|sigla|nome_completo                  |
+-------+-----+-------------------------------+
|P1     |AF   |AFGHANISTAN                    |
|P2     |AL   |ALBANIA                        |
|P3     |null |ALGERIA                        |
|P4     |null |ANGOLA                         |
|P5     |AR   |ARGENTINA                      |
|P6     |null |ARUBA                          |
|P7     |AU   |AUSTRALIA                      |
|P8     |AT   |AUSTRIA                        |
|P9     |null |BAHAMAS                        |
|P10    |null |BELARUS                        |
|P11    |BE   |BELGIUM                        |
|P12    |null |BOLIVIA                        |
|P13    |null |BOSNIA AND HERZEGOVINA         |
|P14    |null |BOTSWANA                       |
|P15    |BR   |BRAZIL                         |
|P16    |null |BULGARIA